In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")

perf = pd.read_sql("""
    SELECT e.match_id, e.tourney_start, e.player_id,
           e.result - e.expected AS r,
           m.surface
    FROM elo_long e
    JOIN matches m ON m.match_id = e.match_id
""", con)
players = pd.read_sql("SELECT player_id, name FROM players", con)

print(perf.shape)
print(perf["surface"].value_counts(dropna=False).to_string())
print(perf["r"].describe().round(3).to_string())

In [ ]:
def form(as_of, surface, window_days=1095, min_matches=10, max_gap_days=182, top=20):
    start = (pd.Timestamp(as_of) - pd.Timedelta(days=window_days)).strftime("%Y-%m-%d")
    d = perf[(perf["surface"] == surface)
             & (perf["tourney_start"] <= as_of)
             & (perf["tourney_start"] >= start)].copy()

    d = d.sort_values("match_id", ascending=False)
    d["i"] = d.groupby("player_id").cumcount()
    d["w"] = 0.5 ** (d["i"] / 10)
    d["w2"] = d["w"] ** 2
    d["wr"] = d["w"] * d["r"]

    g = d.groupby("player_id").agg(matches=("r", "size"), sw=("w", "sum"), sw2=("w2", "sum"),
                                   swr=("wr", "sum"), last_start=("tourney_start", "max"))
    g["form"] = g["swr"] / g["sw"]
    g["n_eff"] = g["sw"] ** 2 / g["sw2"]

    d = d.join(g["form"], on="player_id")
    d["res2"] = (d["w"] * (d["r"] - d["form"])) ** 2
    g["se"] = np.sqrt(d.groupby("player_id")["res2"].sum()) / g["sw"]

    cutoff = (pd.Timestamp(as_of) - pd.Timedelta(days=max_gap_days)).strftime("%Y-%m-%d")
    g = g[(g["matches"] >= min_matches) & (g["last_start"] >= cutoff)]
    g = g.sort_values("form", ascending=False).head(top)

    g = g.join(players.set_index("player_id")["name"])
    g["form_pct"] = (100 * g["form"]).round(1)
    g["plus_minus"] = (196 * g["se"]).round(1)
    g["n_eff"] = g["n_eff"].round(1)
    g.insert(0, "rank", range(1, len(g) + 1))
    return g[["rank", "name", "matches", "n_eff", "form_pct", "plus_minus"]]

for s in ["Clay", "Hard", "Grass"]:
    print(s)
    print(form("2024-12-22", s).to_string(index=False))
    print()

In [ ]:
print(pd.read_sql("SELECT MAX(tourney_start) AS latest, COUNT(*) AS matches FROM matches", con).to_string(index=False))

In [ ]:
def form_table(as_of, surface, **kw):
    t = form(as_of, surface, top=1000, **kw)
    t["lower"] = (t["form_pct"] - t["plus_minus"]).round(1)
    t["upper"] = (t["form_pct"] + t["plus_minus"]).round(1)
    t["evidence"] = np.where(t["lower"] > 0, "above expectation",
                    np.where(t["upper"] < 0, "below expectation", "not enough evidence"))
    return t

full = form_table("2024-12-22", "Clay")
print(full.head(20)[["rank", "name", "matches", "form_pct", "lower", "upper", "evidence"]].to_string(index=False))
print(full["evidence"].value_counts().to_string())

In [ ]:
print(pd.read_sql("""
    SELECT (SELECT COUNT(*) FROM matches) AS matches,
           (SELECT COUNT(*) FROM elo_long) / 2 AS elo_matches,
           (SELECT MAX(match_id) FROM matches) AS max_id_matches,
           (SELECT MAX(match_id) FROM elo_long) AS max_id_elo
""", con).to_string(index=False))

In [ ]:
res, final_rating, final_played = run_elo(games)

a = res[["match_id", "tourney_start", "winner_id", "loser_id", "r_winner", "r_winner_after", "exp_w"]].copy()
a.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "elo_before", "elo_after", "expected"]
a["result"] = 1

b = res[["match_id", "tourney_start", "loser_id", "winner_id", "r_loser", "r_loser_after", "exp_w"]].copy()
b.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "elo_before", "elo_after", "expected"]
b["expected"] = 1 - b["expected"]
b["result"] = 0

elo_long = pd.concat([a, b]).sort_values(["match_id", "result"], ascending=[True, False]).reset_index(drop=True)
print(elo_long.shape)

In [ ]:
print("run_elo" in globals(), "games" in globals())